In [2]:
import json
import pandas as pd
import numpy as np
import re
from datetime import datetime

In [4]:
file_path = "/Users/alyssanguyen/Desktop/IRLE_scraping/scripts/raw_prices_ubereats_nonca_fflocal_090172024.csv"
ca_ff = pd.read_csv(file_path)
file_path_2 = "/Users/alyssanguyen/Desktop/IRLE_scraping/csv_files/uszips.csv"
ca_zip_count = pd.read_csv(file_path_2)
# file_path_3 = "/Users/alyssanguyen/Desktop/IRLE_scraping/csv_files/processed_prices_ubereats_ca_ff_03222024.csv"
# example = pd.read_csv(file_path_3)

In [5]:
#Drop all the columns we don't need 
ca_ff_ = ca_ff.drop(columns=['Unnamed: 0', 'inputted_location','restaurant_distance'])
ca_ff_ = ca_ff_.dropna(subset=['restaurant_location'])

In [6]:
ca_zip_count = ca_zip_count[['zip', 'county_name']]

In [7]:
#restaurant_rating cleaning 
# Ensure the column is of string type using .loc
ca_ff_.loc[:, 'restaurant_rating'] = ca_ff_['restaurant_rating'].astype(str)

# Count rows containing 'mi'
rows_with_mi = ca_ff_['restaurant_rating'].str.contains('mi').sum()
print("Number of rows with 'mi' in restaurant rating:", rows_with_mi)

# Replace invalid ratings ending with 'mi' with '0' using .loc
ca_ff_.loc[:, 'restaurant_rating'] = ca_ff_['restaurant_rating'].str.replace(r'.*mi$', '0', regex=True)

Number of rows with 'mi' in restaurant rating: 4071


In [9]:
#converting data types 
ca_ff_['restaurant_name'] = ca_ff_['restaurant_name'].astype('string')
ca_ff_['menu_item'] = ca_ff_['menu_item'].astype('string')
ca_ff_['menu_item'] = ca_ff_['menu_item'].str.replace(r'\s+', ' ', regex=True)
ca_ff_['restaurant_location'] = ca_ff_['restaurant_location'].astype('string')
ca_ff_['restaurant_rating'] = pd.to_numeric(ca_ff_['restaurant_rating'].str.strip(), errors='coerce')


In [10]:
#cleaning up string columns 

ca_ff_['menu_item'] = ca_ff_['menu_item'].str.lower()
ca_ff_['restaurant_location'] = ca_ff_['restaurant_location'].str.lower()
ca_ff_['restaurant_name'] = ca_ff_['restaurant_name'].str.replace('_', ' ')

#remove special characters
ca_ff_['menu_item'] = ca_ff_['menu_item'].apply(lambda x: ''.join(ch for ch in x if ch.isalnum() or ch.isspace()))
ca_ff_#cleaning up string columns 

ca_ff_['menu_item'] = ca_ff_['menu_item'].str.lower()
ca_ff_['restaurant_location'] = ca_ff_['restaurant_location'].str.lower()
ca_ff_['restaurant_name'] = ca_ff_['restaurant_name'].str.replace('_', ' ')

#remove special characters
ca_ff_['menu_item'] = ca_ff_['menu_item'].apply(lambda x: ''.join(ch for ch in x if ch.isalnum() or ch.isspace()))
ca_ff_['restaurant_name'].unique()

<StringArray>
[            "Baker's Burgers", "Barney's Gourmet Hamburgers",
               'Burger Lounge',               "Fred's Burger",
               'Golden Burger',             "Gott's Roadside",
                'Juicy Burger',              'Mooyah Burgers',
   "Nation's Giant Hamburgers",        'Roam Artisan Burgers',
                  'Trueburger',         'Super Duper Burgers',
          "Baker's Drive-Thru"]
Length: 13, dtype: string

In [11]:
def price_list(x):
    return list(x)

def mean_non_zero(x):
    return np.mean(x[x != 0]) if np.any(x != 0) else 0

def median_non_zero(x):
    return np.median(x[x != 0]) if np.any(x != 0) else 0

def std_non_zero(x):
    return np.std(x[x != 0]) if np.any(x != 0) else 0

In [12]:
#"Barney's Gourmet Hamburgers"

ca_ff_barneys = ca_ff_[ca_ff_['restaurant_name'] == "Barney's Gourmet Hamburgers"]

#First part of grouping 

agg_funcs = {
    'menu_item_price': [mean_non_zero, median_non_zero, std_non_zero],  # calculate the average, median, and standard dev PRICE
    'restaurant_rating': 'mean', # calculate the average RATING 
    'menu_item' : 'count',
    'number_of_ratings': 'first'
}

grouped_barneys = ca_ff_barneys.groupby(['restaurant_name','restaurant_location']).agg(agg_funcs).reset_index()
grouped_barneys.columns = [' '.join(col).strip() for col in grouped_barneys.columns.values]

In [13]:
#Second part of grouping 
barneys_lst = ['barneys burger','single steak fries']

# Filter rows where 'menu_item' contains any item in mcd_lst
menu_items_barneys = ca_ff_barneys[ca_ff_barneys['menu_item'].isin(barneys_lst)].sort_values('menu_item')
menu_items_barneys = menu_items_barneys.drop_duplicates(subset=['restaurant_name', 'restaurant_location', 'menu_item'])

grouped_barneys_2 = menu_items_barneys.groupby(['restaurant_name', 'restaurant_location'])['menu_item_price'].agg(price_list).reset_index()

grouped_barneys_2[['burger', 'fries']] = grouped_barneys_2['menu_item_price'].apply(pd.Series)
grouped_barneys_2.drop(columns=['menu_item_price'], inplace=True)

#Merging the grouped dfs together 
merged_barneys = pd.merge(grouped_barneys, grouped_barneys_2, on=['restaurant_name', 'restaurant_location'], how='inner')
merged_barneys['combo'] = np.nan
merged_barneys['specialty_item'] = np.nan
merged_barneys['cheeseburger'] = np.nan


merged_barneys

,restaurant_name,restaurant_location,menu_item_price mean_non_zero,menu_item_price median_non_zero,menu_item_price std_non_zero,restaurant_rating mean,menu_item count,number_of_ratings first,burger,fries,combo,specialty_item,cheeseburger
0,Barney's Gourmet Hamburgers,"1591 solano ave, berkeley, ca, 94707, us",12.423554,9.08,7.179384,4.7,726,900+,18.15,5.23,NaN,NaN,NaN
1,Barney's Gourmet Hamburgers,"1600 shattuck ave, berkeley, ca, 94709, us",12.550083,9.35,7.065129,4.7,605,900+,18.15,5.23,NaN,NaN,NaN
2,Barney's Gourmet Hamburgers,"4138 24th st, san francisco, ca, 94114, us",12.844138,10.50,7.061680,4.7,870,2,17.55,6.95,NaN,NaN,NaN
3,Barney's Gourmet Hamburgers,"4162 piedmont ave, oakland, ca, 94611, us",12.658992,9.95,6.799162,4.6,903,1,17.50,5.75,NaN,NaN,NaN
4,Barney's Gourmet Hamburgers,"4776 commons way ste d, calabasas, ca, 91302, us",11.455660,9.00,6.571380,4.6,212,240+,16.25,4.95,NaN,NaN,NaN
5,Barney's Gourmet Hamburgers,"5819 college ave, oakland, ca, 94618, us",11.658154,8.75,7.427733,4.6,780,500+,18.15,5.23,NaN,NaN,NaN


In [14]:
#'Burger Lounge'

ca_ff_bl = ca_ff_[ca_ff_['restaurant_name'] == 'Burger Lounge']

#First part of grouping 

agg_funcs = {
    'menu_item_price': [mean_non_zero, median_non_zero, std_non_zero],  # calculate the average, median, and standard dev PRICE
    'restaurant_rating': 'mean', # calculate the average RATING 
    'menu_item' : 'count',
    'number_of_ratings': 'first'
}

grouped_bl = ca_ff_bl.groupby(['restaurant_name','restaurant_location']).agg(agg_funcs).reset_index()
grouped_bl.columns = [' '.join(col).strip() for col in grouped_bl.columns.values]

In [15]:
#Second part of grouping 
bl_lst = ['regular fries','the classic', ]

# Filter rows where 'menu_item' contains any item in mcd_lst
menu_items_bl = ca_ff_bl[ca_ff_bl['menu_item'].isin(bl_lst)].sort_values('menu_item')
menu_items_bl = menu_items_bl.drop_duplicates(subset=['restaurant_name', 'restaurant_location', 'menu_item'])

grouped_bl_2 = menu_items_bl.groupby(['restaurant_name', 'restaurant_location'])['menu_item_price'].agg(price_list).reset_index()

grouped_bl_2[['fries', 'cheeseburger']] = grouped_bl_2['menu_item_price'].apply(pd.Series)
grouped_bl_2.drop(columns=['menu_item_price'], inplace=True)

#Merging the grouped dfs together 
merged_bl = pd.merge(grouped_bl, grouped_bl_2, on=['restaurant_name', 'restaurant_location'], how='inner')
merged_bl['combo'] = np.nan
merged_bl['specialty_item'] = np.nan
merged_bl['hamburger'] = np.nan


merged_bl

,restaurant_name,restaurant_location,menu_item_price mean_non_zero,menu_item_price median_non_zero,menu_item_price std_non_zero,restaurant_rating mean,menu_item count,number_of_ratings first,fries,cheeseburger,combo,specialty_item,hamburger
0,Burger Lounge,"1198, roseville, ca, 95678, us",10.781471,10.74,6.394666,4.6,280,310+,5.94,13.14,NaN,NaN,NaN
1,Burger Lounge,"13455 maxella ave, marina del rey, ca, 90292, us",10.524000,10.74,4.577872,4.6,67,2,5.94,13.14,NaN,NaN,NaN
2,Burger Lounge,"1608 india st, san diego, ca, 92101, us",11.267826,10.74,6.169317,4.6,142,2,5.94,13.14,NaN,NaN,NaN
3,Burger Lounge,"16490 paseo del sur, san diego, ca, 92127, us",11.083944,10.74,6.160525,4.6,73,1,5.94,13.14,NaN,NaN,NaN
4,Burger Lounge,"1875 s bascom ave, campbell, ca, 95008, us",10.936324,10.74,6.367456,4.6,280,900+,5.94,13.14,NaN,NaN,NaN
5,Burger Lounge,"19515 nordhoff st., ste 5, northridge, ca, 913...",11.123143,10.74,6.218773,4.7,72,700+,5.94,13.14,NaN,NaN,NaN
6,Burger Lounge,"213 arizona ave, santa monica, ca, 90401, us",11.146364,10.74,6.342088,4.6,68,1,5.94,13.14,NaN,NaN,NaN
7,Burger Lounge,"217 n larchmont blvd, los angeles, ca, 90004, us",11.431452,10.74,6.357325,4.7,128,5,5.94,13.14,NaN,NaN,NaN
8,Burger Lounge,"2720 via de la valle, del mar, ca, 92014, us",11.209242,10.74,6.272355,4.7,68,1,5.94,13.14,NaN,NaN,NaN
9,Burger Lounge,"279 e 17th st, costa mesa, ca, 92627, us",11.050746,10.74,6.325365,4.7,138,2,5.94,13.14,NaN,NaN,NaN


In [16]:
#'Roam Artisan Burgers'

ca_ff_roam = ca_ff_[ca_ff_['restaurant_name'] == 'Roam Artisan Burgers']

#First part of grouping 

agg_funcs = {
    'menu_item_price': [mean_non_zero, median_non_zero, std_non_zero],  # calculate the average, median, and standard dev PRICE
    'restaurant_rating': 'mean', # calculate the average RATING 
    'menu_item' : 'count',
    'number_of_ratings': 'first'
}

grouped_roam = ca_ff_roam.groupby(['restaurant_name','restaurant_location']).agg(agg_funcs).reset_index()
grouped_roam.columns = [' '.join(col).strip() for col in grouped_roam.columns.values]

In [17]:
#Second part of grouping 
roam_lst = ['russet fries', 'the classic burger' ]

# Filter rows where 'menu_item' contains any item in mcd_lst
menu_items_roam = ca_ff_roam[ca_ff_roam['menu_item'].isin(roam_lst)].sort_values('menu_item')
menu_items_roam = menu_items_roam.drop_duplicates(subset=['restaurant_name', 'restaurant_location', 'menu_item'])

grouped_roam_2 = menu_items_roam.groupby(['restaurant_name', 'restaurant_location'])['menu_item_price'].agg(price_list).reset_index()

grouped_roam_2[['fries', 'hamburger']] = grouped_roam_2['menu_item_price'].apply(pd.Series)
grouped_roam_2.drop(columns=['menu_item_price'], inplace=True)

#Merging the grouped dfs together 
merged_roam = pd.merge(grouped_roam, grouped_roam_2, on=['restaurant_name', 'restaurant_location'], how='inner')
merged_roam['combo'] = np.nan
merged_roam['specialty_item'] = np.nan
merged_roam['cheeseburger'] = np.nan


merged_roam

,restaurant_name,restaurant_location,menu_item_price mean_non_zero,menu_item_price median_non_zero,menu_item_price std_non_zero,restaurant_rating mean,menu_item count,number_of_ratings first,fries,hamburger,combo,specialty_item,cheeseburger
0,Roam Artisan Burgers,"1785 union st, san francisco, ca, 94123, us",13.541644,10.35,12.398395,4.8,711,2,5.75,14.38,NaN,NaN,NaN
1,Roam Artisan Burgers,"1923 fillmore st, san francisco, ca, 94115, us",13.617917,10.35,12.467178,4.7,693,4,5.75,14.38,NaN,NaN,NaN
2,Roam Artisan Burgers,"23, lafayette, ca, 94549, us",13.541644,10.35,12.398395,4.7,316,700+,5.75,14.38,NaN,NaN,NaN
3,Roam Artisan Burgers,"3081 s delaware st, san mateo, ca, 94403, us",13.541644,10.35,12.398395,4.6,316,500+,5.75,14.38,NaN,NaN,NaN
4,Roam Artisan Burgers,"6000 bollinger canyon road, san ramon, ca, 945...",13.541644,10.35,12.398395,4.6,237,350+,5.75,14.38,NaN,NaN,NaN


In [18]:
#'Super Duper Burgers'

ca_ff_super = ca_ff_[ca_ff_['restaurant_name'] == 'Super Duper Burgers']

#First part of grouping 

agg_funcs = {
    'menu_item_price': [mean_non_zero, median_non_zero, std_non_zero],  # calculate the average, median, and standard dev PRICE
    'restaurant_rating': 'mean', # calculate the average RATING 
    'menu_item' : 'count',
    'number_of_ratings': 'first'
}

grouped_super = ca_ff_super.groupby(['restaurant_name','restaurant_location']).agg(agg_funcs).reset_index()
grouped_super.columns = [' '.join(col).strip() for col in grouped_super.columns.values]

In [19]:
#Second part of grouping 
super_lst = ['french fries', 'super burger' ]

# Filter rows where 'menu_item' contains any item in mcd_lst
menu_items_super = ca_ff_super[ca_ff_super['menu_item'].isin(super_lst)].sort_values('menu_item')
menu_items_super = menu_items_super.drop_duplicates(subset=['restaurant_name', 'restaurant_location', 'menu_item'])

grouped_super_2 = menu_items_super.groupby(['restaurant_name', 'restaurant_location'])['menu_item_price'].agg(price_list).reset_index()

grouped_super_2[['fries', 'hamburger']] = grouped_super_2['menu_item_price'].apply(pd.Series)
grouped_super_2.drop(columns=['menu_item_price'], inplace=True)

#Merging the grouped dfs together 
merged_super = pd.merge(grouped_super, grouped_super_2, on=['restaurant_name', 'restaurant_location'], how='inner')
merged_super['combo'] = np.nan
merged_super['specialty_item'] = np.nan
merged_super['cheeseburger'] = np.nan

merged_super

,restaurant_name,restaurant_location,menu_item_price mean_non_zero,menu_item_price median_non_zero,menu_item_price std_non_zero,restaurant_rating mean,menu_item count,number_of_ratings first,fries,hamburger,combo,specialty_item,cheeseburger
0,Super Duper Burgers,"1100 park place, san mateo, ca, 94403, us",6.456897,5.25,2.905901,4.6,136,56,4.00,10.5,NaN,NaN,NaN
1,Super Duper Burgers,"1168 galleria blvd, roseville, ca, 95678, us",10.232000,12.99,4.857035,4.5,16,48,2.00,NaN,NaN,NaN,NaN
2,Super Duper Burgers,"127 serramonte center, daly city, ca, 94015, us",6.383929,5.25,2.931116,4.8,132,600+,4.00,10.5,NaN,NaN,NaN
3,Super Duper Burgers,"15991, los gatos, ca, 95032, us",6.383929,5.25,2.931116,4.6,99,170+,4.00,10.5,NaN,NaN,NaN
4,Super Duper Burgers,"19252 soledad canyon road, santa clarita, ca, ...",10.098667,12.49,4.878273,4.4,16,170+,2.00,NaN,NaN,NaN,NaN
5,Super Duper Burgers,"1988 n main st, salinas, ca, 93906, us",10.232000,12.99,4.857035,3.8,16,47,2.00,NaN,NaN,NaN,NaN
6,Super Duper Burgers,"2003 diamond blvd ste 100, concord, ca, 94520, us",6.134615,5.00,2.878326,4.7,93,110+,4.00,10.5,NaN,NaN,NaN
7,Super Duper Burgers,"2145 ventura blvd, camarillo, ca, 93010, us",8.482323,7.50,6.024094,4.7,109,390+,6.25,NaN,NaN,NaN,NaN
8,Super Duper Burgers,"2201 chestnut street, san francisco, ca, 94123...",6.134615,5.00,2.878326,4.8,186,700+,4.00,10.5,NaN,NaN,NaN
9,Super Duper Burgers,"2355 telegraph avenue, berkeley, ca, 94704, us",6.344828,5.25,2.887558,4.6,68,500+,4.00,10.5,NaN,NaN,NaN


In [20]:
#'Juicy Burger'

ca_ff_juicy = ca_ff_[ca_ff_['restaurant_name'] == 'Juicy Burger']

#First part of grouping 

agg_funcs = {
    'menu_item_price': [mean_non_zero, median_non_zero, std_non_zero],  # calculate the average, median, and standard dev PRICE
    'restaurant_rating': 'mean', # calculate the average RATING 
    'menu_item' : 'count',
    'number_of_ratings': 'first'
}

grouped_juicy = ca_ff_juicy.groupby(['restaurant_name','restaurant_location']).agg(agg_funcs).reset_index()
grouped_juicy.columns = [' '.join(col).strip() for col in grouped_juicy.columns.values]

In [21]:
ca_ff_juicy['menu_item'].value_counts().head(40)

menu_item
new york style cheesecake            24
onion rings                          22
sprite                               17
diet coke                            17
grilled chicken sandwich             16
dasani bottled water                 16
dr pepper                            16
milk                                 15
ultimate cheeseburger                14
homestyle ranch chicken club         14
barqs root beer                      14
6pc classic french toast sticks      14
ultimate breakfast sandwich          14
loaded breakfast sandwich            14
jumbo jack cheeseburger              14
bacon ultimate cheeseburger          14
classic crispy jack wrap             14
supreme croissant                    14
double jack                          14
bacon double smashed jack            14
sausage croissant                    14
hash brown                           14
classic smashed jack                 14
sourdough jack                       14
8pc chicken nuggets           

In [26]:
#Second part of grouping 
juicy_lst = ['ultimate cheeseburger']

# Filter rows where 'menu_item' contains any item in mcd_lst
menu_items_juicy = ca_ff_juicy[ca_ff_juicy['menu_item'].isin(juicy_lst)].sort_values('menu_item')
menu_items_juicy = menu_items_juicy.drop_duplicates(subset=['restaurant_name', 'restaurant_location', 'menu_item'])

grouped_juicy_2 = menu_items_juicy.groupby(['restaurant_name', 'restaurant_location'])['menu_item_price'].agg(price_list).reset_index()

grouped_juicy_2[['cheeseburger']] = grouped_juicy_2['menu_item_price'].apply(pd.Series)
grouped_juicy_2.drop(columns=['menu_item_price'], inplace=True)

#Merging the grouped dfs together 
merged_juicy = pd.merge(grouped_juicy, grouped_juicy_2, on=['restaurant_name', 'restaurant_location'], how='inner')
merged_juicy['combo'] = np.nan
merged_juicy['specialty_item'] = np.nan
merged_juicy['hamburger'] = np.nan
merged_juicy['fries'] = np.nan


merged_juicy

,restaurant_name,restaurant_location,menu_item_price mean_non_zero,menu_item_price median_non_zero,menu_item_price std_non_zero,restaurant_rating mean,menu_item count,number_of_ratings first,cheeseburger,combo,specialty_item,hamburger,fries
0,Juicy Burger,"148 e san carlos st, san jose, ca, 95112, us",10.250811,8.740,5.481347,4.5,185,2,10.49,NaN,NaN,NaN,NaN
1,Juicy Burger,"1504 pacific ave, stockton, ca, 95204, us",10.315082,8.740,5.786135,4.2,183,500+,10.49,NaN,NaN,NaN,NaN
2,Juicy Burger,"1900 mission ave, oceanside, ca, 92054, us",9.729148,7.830,5.591545,4.6,176,1,10.36,NaN,NaN,NaN,NaN
3,Juicy Burger,"2220 chester ave, bakersfield, ca, 93301, us",9.912757,8.490,5.524798,4.4,185,300+,10.24,NaN,NaN,NaN,NaN
4,Juicy Burger,"24620 madison ave, murrieta, ca, 92562, us",9.844199,8.110,5.578493,4.4,181,340+,10.24,NaN,NaN,NaN,NaN
5,Juicy Burger,"3434 14th st, riverside, ca, 92501, us",9.230270,8.500,4.669064,4.3,186,500+,10.36,NaN,NaN,NaN,NaN
6,Juicy Burger,"400 broadway st., vallejo, ca, 94590, us",10.068770,8.490,5.702686,4.4,187,500+,10.36,NaN,NaN,NaN,NaN
7,Juicy Burger,"495 n d st, san bernardino, ca, 92401, us",9.142882,7.460,5.518574,4.5,170,900+,8.74,NaN,NaN,NaN,NaN
8,Juicy Burger,"601 north main street, santa ana, ca, 92701, us",9.109027,7.800,5.377137,4.4,185,800+,8.49,NaN,NaN,NaN,NaN
9,Juicy Burger,"640 ocean street, santa cruz, ca, 95060, us",9.665430,8.110,5.507436,4.4,186,3,9.99,NaN,NaN,NaN,NaN


In [27]:
#'Golden Burger'

ca_ff_golden = ca_ff_[ca_ff_['restaurant_name'] == 'Golden Burger']

#First part of grouping 

agg_funcs = {
    'menu_item_price': [mean_non_zero, median_non_zero, std_non_zero],  # calculate the average, median, and standard dev PRICE
    'restaurant_rating': 'mean', # calculate the average RATING 
    'menu_item' : 'count',
    'number_of_ratings': 'first'
}

grouped_golden = ca_ff_golden.groupby(['restaurant_name','restaurant_location']).agg(agg_funcs).reset_index()
grouped_golden.columns = [' '.join(col).strip() for col in grouped_golden.columns.values]

In [28]:
#Second part of grouping 
golden_lst = ['cheeseburger', 'french fries', 'hamburger']

# Filter rows where 'menu_item' contains any item in mcd_lst
menu_items_golden = ca_ff_golden[ca_ff_golden['menu_item'].isin(golden_lst)].sort_values('menu_item')
menu_items_golden = menu_items_golden.drop_duplicates(subset=['restaurant_name', 'restaurant_location', 'menu_item'])

grouped_golden_2 = menu_items_golden.groupby(['restaurant_name', 'restaurant_location'])['menu_item_price'].agg(price_list).reset_index()

grouped_golden_2[['cheeseburger', 'fries', 'hamburger']] = grouped_golden_2['menu_item_price'].apply(pd.Series)
grouped_golden_2.drop(columns=['menu_item_price'], inplace=True)

#Merging the grouped dfs together 
merged_golden = pd.merge(grouped_golden, grouped_golden_2, on=['restaurant_name', 'restaurant_location'], how='inner')
merged_golden['combo'] = np.nan
merged_golden['specialty_item'] = np.nan


merged_golden

,restaurant_name,restaurant_location,menu_item_price mean_non_zero,menu_item_price median_non_zero,menu_item_price std_non_zero,restaurant_rating mean,menu_item count,number_of_ratings first,cheeseburger,fries,hamburger,combo,specialty_item
0,Golden Burger,"1010 a st, hayward, ca, 94541, us",4.347949,4.83,1.607065,4.5,50,800+,4.46,3.15,NaN,NaN,NaN
1,Golden Burger,"1428 polk street, san francisco, ca, 94109, us",12.870435,14.49,5.630068,4.2,46,87,8.99,NaN,NaN,NaN,NaN
2,Golden Burger,"3846 mowry avenue, fremont, ca, 94538, us",8.540000,6.99,5.572028,3.4,20,20,12.99,6.99,10.99,NaN,NaN
3,Golden Burger,"7504 mission grove pkwy s, riverside, ca, 9250...",10.122973,8.70,4.553386,4.7,37,280+,6.90,5.50,5.90,NaN,NaN


In [29]:
#"Gott's Roadside"

ca_ff_gotts = ca_ff_[ca_ff_['restaurant_name'] == "Gott's Roadside"]

#First part of grouping 

agg_funcs = {
    'menu_item_price': [mean_non_zero, median_non_zero, std_non_zero],  # calculate the average, median, and standard dev PRICE
    'restaurant_rating': 'mean', # calculate the average RATING 
    'menu_item' : 'count',
    'number_of_ratings': 'first'
}

grouped_gotts = ca_ff_gotts.groupby(['restaurant_name','restaurant_location']).agg(agg_funcs).reset_index()
grouped_gotts.columns = [' '.join(col).strip() for col in grouped_gotts.columns.values]

In [30]:
#Second part of grouping 
gotts_lst = ['cheeseburger', 'fries', 'hamburger']

# Filter rows where 'menu_item' contains any item in mcd_lst
menu_items_gotts = ca_ff_gotts[ca_ff_gotts['menu_item'].isin(gotts_lst)].sort_values('menu_item')
menu_items_gotts = menu_items_gotts.drop_duplicates(subset=['restaurant_name', 'restaurant_location', 'menu_item'])

grouped_gotts_2 = menu_items_gotts.groupby(['restaurant_name', 'restaurant_location'])['menu_item_price'].agg(price_list).reset_index()

grouped_gotts_2[['cheeseburger', 'fries', 'hamburger']] = grouped_gotts_2['menu_item_price'].apply(pd.Series)
grouped_gotts_2.drop(columns=['menu_item_price'], inplace=True)

#Merging the grouped dfs together 
merged_gotts = pd.merge(grouped_gotts, grouped_gotts_2, on=['restaurant_name', 'restaurant_location'], how='inner')
merged_gotts['combo'] = np.nan
merged_gotts['specialty_item'] = np.nan


merged_gotts

,restaurant_name,restaurant_location,menu_item_price mean_non_zero,menu_item_price median_non_zero,menu_item_price std_non_zero,restaurant_rating mean,menu_item count,number_of_ratings first,cheeseburger,fries,hamburger,combo,specialty_item
0,Gott's Roadside,"1 ferry building, san francisco, ca, 94111, us",19.211870,15.99,16.152819,4.8,441,4,10.99,5.29,10.99,NaN,NaN
1,Gott's Roadside,"1200, roseville, ca, 95678, us",17.502412,17.45,12.869995,4.4,228,1,5.95,NaN,NaN,NaN,NaN
2,Gott's Roadside,"1275 south main street, walnut creek, ca, 9459...",18.505573,14.99,15.716661,4.6,780,370+,10.99,4.99,10.99,NaN,NaN
3,Gott's Roadside,"151 warriors way, san francisco, ca, 94158, us",19.046899,15.99,15.716161,4.8,612,2,10.99,5.29,10.99,NaN,NaN
4,Gott's Roadside,"22920 centerpoint drive, moreno valley, ca, 92...",17.485022,17.25,12.874140,4.5,227,1,5.95,NaN,NaN,NaN,NaN
5,Gott's Roadside,"24320 town center drive, valencia, ca, 91355, us",17.755044,17.45,12.774370,4.4,228,2,5.95,NaN,NaN,NaN,NaN
6,Gott's Roadside,"26500 ynez road, temecula, ca, 92591, us",17.460129,17.25,12.744827,4.5,232,1,5.95,NaN,NaN,NaN,NaN
7,Gott's Roadside,"302 bon air center, larkspur, ca, 94904, us",18.505659,14.99,15.802625,4.8,308,800+,10.99,4.99,10.99,NaN,NaN
8,Gott's Roadside,"644 1st st, napa, ca, 94559, us",18.594077,14.99,15.744441,4.7,462,430+,10.99,4.99,10.99,NaN,NaN
9,Gott's Roadside,"9237 laguna springs drive, elk grove, ca, 9575...",17.595852,17.45,12.814921,4.2,229,1,5.95,NaN,NaN,NaN,NaN


In [31]:
#"Nation's Giant Hamburgers"

ca_ff_nations = ca_ff_[ca_ff_['restaurant_name'] == "Nation's Giant Hamburgers"]

#First part of grouping 

agg_funcs = {
    'menu_item_price': [mean_non_zero, median_non_zero, std_non_zero],  # calculate the average, median, and standard dev PRICE
    'restaurant_rating': 'mean', # calculate the average RATING 
    'menu_item' : 'count',
    'number_of_ratings': 'first'
}

grouped_nations = ca_ff_nations.groupby(['restaurant_name','restaurant_location']).agg(agg_funcs).reset_index()
grouped_nations.columns = [' '.join(col).strip() for col in grouped_nations.columns.values]

In [32]:
#Second part of grouping 
nations_lst = ['cheeseburger', 'french fries', 'hamburger']

# Filter rows where 'menu_item' contains any item in mcd_lst
menu_items_nations = ca_ff_nations[ca_ff_nations['menu_item'].isin(nations_lst)].sort_values('menu_item')
menu_items_nations = menu_items_nations.drop_duplicates(subset=['restaurant_name', 'restaurant_location', 'menu_item'])

grouped_nations_2 = menu_items_nations.groupby(['restaurant_name', 'restaurant_location'])['menu_item_price'].agg(price_list).reset_index()

grouped_nations_2[['cheeseburger', 'fries', 'hamburger']] = grouped_nations_2['menu_item_price'].apply(pd.Series)
grouped_nations_2.drop(columns=['menu_item_price'], inplace=True)

#Merging the grouped dfs together 
merged_nations = pd.merge(grouped_nations, grouped_nations_2, on=['restaurant_name', 'restaurant_location'], how='inner')
merged_nations['combo'] = np.nan
merged_nations['specialty_item'] = np.nan


merged_nations

,restaurant_name,restaurant_location,menu_item_price mean_non_zero,menu_item_price median_non_zero,menu_item_price std_non_zero,restaurant_rating mean,menu_item count,number_of_ratings first,cheeseburger,fries,hamburger,combo,specialty_item
0,Nation's Giant Hamburgers,"127 west 4th street, long beach, ca, 90802, us",5.615467,4.280,2.758592,4.5,118,1,3.50,5.19,2.46,NaN,NaN
1,Nation's Giant Hamburgers,"13296, san pablo, ca, 94806, us",11.935275,10.550,6.002926,4.5,321,1,11.40,4.25,9.30,NaN,NaN
2,Nation's Giant Hamburgers,"1335 washington ave, san leandro, ca, 94577, us",12.039286,10.425,6.211207,4.6,392,1,11.30,4.25,9.25,NaN,NaN
3,Nation's Giant Hamburgers,"1424 first st, livermore, ca, 94550, us",11.998131,10.550,6.035239,4.8,111,380+,11.40,4.25,9.30,NaN,NaN
4,Nation's Giant Hamburgers,"1432 webster st, alameda, ca, 94501, us",11.865714,10.550,5.982098,4.6,545,1,11.40,4.25,9.30,NaN,NaN
5,Nation's Giant Hamburgers,"1441 3rd st, napa, ca, 94559, us",11.998131,10.550,6.035239,4.6,222,700+,11.40,4.25,9.30,NaN,NaN
6,Nation's Giant Hamburgers,"15911 pioneer blvd, norwalk, ca, 90650, us",10.122973,8.700,4.553386,4.7,37,1,6.90,5.50,5.90,NaN,NaN
7,Nation's Giant Hamburgers,"16396, san pablo, ca, 94806, us",12.222840,9.900,6.483341,4.6,81,600+,11.30,4.25,9.25,NaN,NaN
8,Nation's Giant Hamburgers,"16984 valley blvd., fontana, ca, 92335, us",7.182396,4.870,4.937008,4.4,96,900+,4.01,NaN,NaN,NaN,NaN
9,Nation's Giant Hamburgers,"1800 university ave, berkeley, ca, 94703, us",12.007979,10.150,6.243227,4.7,188,1,11.30,4.25,9.25,NaN,NaN


In [33]:
#Stack all restaurants
ubereats_fflocal_prices = pd.concat([merged_super, merged_juicy,merged_gotts, merged_nations]).reset_index(drop=True)

In [34]:
pattern = r",\s*([a-zA-Z]{2})\s*,?\s*(\d{5}(?:-\d{4})?)"

def extract_state_zip(address):
    match = re.search(pattern, address)
    if match:
        state, zip_code = match.groups()
        return state, zip_code
    else:
        return None, None

# Apply the function to extract state and zip code
ubereats_fflocal_prices[['state', 'zip']] = ubereats_fflocal_prices['restaurant_location'].apply(lambda x: pd.Series(extract_state_zip(x)))
ubereats_fflocal_prices['zip'] = ubereats_fflocal_prices['zip'].str.split('-').str[0].astype(int)

#Get county 
ubereats_fflocal_prices = ubereats_fflocal_prices.merge(ca_zip_count, on = 'zip')

In [36]:
specific_date = datetime.strptime('09172024', '%m%d%Y')
# Assign the datetime object to the entire 'date' column
ubereats_fflocal_prices['date'] = specific_date
ubereats_fflocal_prices['uber_eats'] = 1
ubereats_fflocal_prices['post_policy'] = 1
ubereats_fflocal_prices['fast_food'] = 1
ubereats_fflocal_prices['local'] = 1

In [38]:
#Save as csv 
ubereats_fflocal_prices.to_csv('processed_prices_ubereats_nonca_fflocal_09172024.csv', index = True)